# Step 4 — Same perovskite benchmark for other model families (CHGNet, SevenNet, MatterSim, ORB)
Each family has its own Python package, and they can conflict, so **run this notebook once per family**: change `FAMILY` in cell 1, then Exécution → Tout exécuter. After each run, restart (Exécution → Déconnecter et supprimer l'environnement) before the next family.

**Run:** Exécution → Modifier le type d'exécution → **GPU T4** → Enregistrer. Then Exécution → **Tout exécuter**.
At the end `step4_<family>.csv` is downloaded. Send it to Claude. Models that fail to load are skipped.

In [ ]:
FAMILY = "orb"     # run 4 times: "chgnet", "sevennet", "mattersim", "orb"
PKG = {"chgnet": "chgnet", "sevennet": "sevenn", "mattersim": "mattersim", "orb": "orb-models"}[FAMILY]
if FAMILY == "orb":   # orb-models pins an old dm-tree without Python 3.13 wheels
    !pip install -q "dm-tree>=0.1.9" cached_path phonopy spglib "ase>=3.23"
    !pip install -q --no-deps "orb-models>=0.5,<0.6"
    import importlib.metadata as _md, subprocess as _sp
    _reqs = [r.split(";")[0].strip() for r in (_md.requires("orb-models") or [])
             if "extra ==" not in r and not r.lower().replace("_", "-").startswith("dm-tree")]
    print("orb-models", _md.version("orb-models"), "| installing dependencies:", _reqs)
    _sp.run(["pip", "install", "-q", *_reqs])
else:
    !pip install -q {PKG} phonopy spglib "ase>=3.23"


In [ ]:
import numpy as np, pandas as pd, torch, time, spglib, ase, traceback
from ase import Atoms
from ase.constraints import FixSymmetry
from ase.filters import FrechetCellFilter
from ase.optimize import BFGS
from phonopy import Phonopy
from phonopy.structure.atoms import PhonopyAtoms
import importlib.metadata as md
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device, "| ase", ase.__version__, "|", PKG, md.version(PKG))

MODELS = {"chgnet": ["chgnet-0.3.0"],
          "sevennet": ["7net-0", "7net-mf-ompa"],
          "mattersim": ["MatterSim-v1.0.0-1M", "MatterSim-v1.0.0-5M"],
          "orb": ["orb-v3-conservative-inf-mpa", "orb-v3-conservative-inf-omat"]}[FAMILY]

def get_calc(name):
    if FAMILY == "chgnet":
        from chgnet.model import CHGNet
        from chgnet.model.dynamics import CHGNetCalculator
        return CHGNetCalculator(CHGNet.load(model_name=name.split("-")[1]), use_device=device)
    if FAMILY == "sevennet":
        from sevenn.calculator import SevenNetCalculator
        if name == "7net-mf-ompa":
            return SevenNetCalculator(model=name, modal="mpa", device=device)
        return SevenNetCalculator(model=name, device=device)
    if FAMILY == "mattersim":
        from mattersim.forcefield import MatterSimCalculator
        return MatterSimCalculator(load_path=name + ".pth", device=device)
    if FAMILY == "orb":
        import pkgutil, importlib, inspect, orb_models
        from orb_models.forcefield import pretrained
        # find ORBCalculator wherever this version of orb-models puts it
        Calc = None
        for mi in pkgutil.walk_packages(orb_models.__path__, "orb_models."):
            if "calculator" not in mi.name: continue
            try: mod = importlib.import_module(mi.name)
            except Exception as ex: print("cannot import", mi.name, repr(ex)[:200]); continue
            if hasattr(mod, "ORBCalculator"): Calc = mod.ORBCalculator; print("ORBCalculator from", mi.name); break
        assert Calc is not None, "ORBCalculator not found"
        fname = name.replace("-", "_")
        avail = [f for f in dir(pretrained) if f.startswith("orb")]
        print("available ORB models:", avail)
        assert fname in avail, f"{fname} not in this orb-models version"
        fn = getattr(pretrained, fname)
        kw = {"device": device}
        if "precision" in inspect.signature(fn).parameters: kw["precision"] = "float32-highest"
        orbff = fn(**kw)
        ckw = {"device": device} if "device" in inspect.signature(Calc).parameters else {}
        return Calc(orbff, **ckw)
SYSTEMS = {"BaTiO3": ("Ba","Ti"), "PbTiO3": ("Pb","Ti"), "KNbO3": ("K","Nb"),
           "SrTiO3": ("Sr","Ti"), "KTaO3": ("K","Ta")}
DIRS = {"Pm-3m": None, "P4mm": [0,0,1], "Amm2": [1,1,0], "R3m": [1,1,1]}
QP = {"G": [0,0,0], "X": [0,0,.5], "M": [.5,.5,0], "R": [.5,.5,.5]}
THz2cm = 33.35641

def perovskite(A, B, a=4.0):
    frac = [[0,0,0],[.5,.5,.5],[.5,.5,0],[.5,0,.5],[0,.5,.5]]
    return Atoms([A,B,"O","O","O"], scaled_positions=frac, cell=np.eye(3)*a, pbc=True)

def distort(at, d, dB=0.10, dO=-0.06):
    at = at.copy(); d = np.array(d, float); d /= np.linalg.norm(d)
    p = at.get_positions(); p[1] += dB*d; p[2:] += dO*d; at.set_positions(p); return at

def spg(at, prec=1e-3):
    return spglib.get_spacegroup((at.cell[:], at.get_scaled_positions(), at.numbers), symprec=prec)

def relax(at, calc, fmax=1e-3, steps=3000):
    at = at.copy(); at.calc = calc
    at.set_constraint(FixSymmetry(at, symprec=1e-4))
    opt = BFGS(FrechetCellFilter(at), logfile=None); conv = opt.run(fmax=fmax, steps=steps)
    at.set_constraint(); return at, bool(conv), opt.nsteps

def phonons(at, calc):
    u = PhonopyAtoms(symbols=at.get_chemical_symbols(), cell=at.cell[:], scaled_positions=at.get_scaled_positions())
    ph = Phonopy(u, supercell_matrix=np.eye(3, dtype=int)*2)
    ph.generate_displacements(distance=0.01)
    F = []
    for sc in ph.supercells_with_displacements:
        s = Atoms(sc.symbols, cell=sc.cell, scaled_positions=sc.scaled_positions, pbc=True); s.calc = calc
        F.append(s.get_forces().copy())
    ph.forces = np.array(F); ph.produce_force_constants()
    r = {}
    for k, q in QP.items():
        f = np.sort(ph.get_frequencies(q))*THz2cm
        nim = int((f < -5).sum())
        if k == "G":   # remove the 3 acoustic modes (closest to zero)
            f = np.delete(f, np.argsort(np.abs(f))[:3])
        r[f"{k}_lowest_cm1"] = f[0]; r[f"{k}_n_imag"] = int((f < -5).sum())
        r[f"{k}_all_cm1"] = " ".join(f"{x:.1f}" for x in f)
    return r

def run_model(MODEL):
    calc = get_calc(MODEL)
    rows = []
    for name, (A, B) in SYSTEMS.items():
        E0 = None
        for ph, d in DIRS.items():
            t0 = time.time()
            start = perovskite(A, B) if d is None else distort(perovskite(A, B), d)
            at, conv, n = relax(start, calc)
            cp = at.cell.cellpar(); E = at.get_potential_energy()
            if ph == "Pm-3m": E0 = E; extra = phonons(at, calc)
            else: extra = {}
            offc = np.linalg.norm(at.get_positions()[1] - (at.get_positions()[0] + at.cell.sum(0)/2))
            rows.append(dict(model=MODEL, material=name, phase=ph, E_eV=E, dE_meV_per_fu=1000*(E-E0),
                             a=cp[0], b=cp[1], c=cp[2], alpha=cp[3], beta=cp[4], gamma=cp[5],
                             c_over_a=cp[2]/cp[0], volume=at.get_volume(), B_offcentre_A=offc,
                             spg_final=spg(at), spg_loose=spg(at, 1e-2), converged=conv, steps=n,
                             maxF=np.abs(at.get_forces()).max(), time_s=round(time.time()-t0, 1), **extra))
            print(f"{MODEL:22s} {name:7s} {ph:6s} dE={rows[-1]['dE_meV_per_fu']:8.2f} meV  c/a={cp[2]/cp[0]:.4f}  {rows[-1]['spg_final']}")
    return pd.DataFrame(rows)


In [ ]:
all_df, failed = [], {}
for m in MODELS:
    try:
        t0 = time.time(); all_df.append(run_model(m)); print(f"--- {m} done in {time.time()-t0:.0f} s")
    except Exception as e:
        failed[m] = repr(e); print("FAILED", m, repr(e)[:300])
    torch.cuda.empty_cache()
res = pd.concat(all_df, ignore_index=True)
res["package_version"] = md.version(PKG)
print("failed:", failed)
res.pivot_table(index=["material","phase"], columns="model", values="dE_meV_per_fu").round(1)


In [ ]:
from google.colab import files
fn = f"step4_{FAMILY}.csv"; res.to_csv(fn, index=False)
pd.Series(failed, dtype=str).to_csv(f"step4_{FAMILY}_failed.csv")
files.download(fn); print("saved", fn)
